In [12]:
from google.colab import drive
drive.mount("/content/drive")

# CHANGE this to your repo folder
%cd /content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications
!ls

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications
build		    experiments  pyproject.toml  src	     test.sh
build_and_test.sh   LICENSE	 README.md	 SUPPORT.md
CODE_OF_CONDUCT.md  pipelines	 SECURITY.md	 tests


In [ ]:
!pip install -e .

In [1]:
import slicegpt
from slicegpt import rotate, model_utils, hf_utils
print("slicegpt imported:", slicegpt.__file__)
print("rotate imported:", rotate.__file__)
print("model_utils imported:", model_utils.__file__)
print("hf_utils_utils imported:", hf_utils.__file__)

slicegpt imported: /content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications/src/slicegpt/__init__.py
rotate imported: /content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications/src/slicegpt/rotate.py
model_utils imported: /content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications/src/slicegpt/model_utils.py
hf_utils_utils imported: /content/drive/MyDrive/TUM/Pratikum/SliceGPTModifications/src/slicegpt/hf_utils.py


In [ ]:
!pip install evaluate

In [15]:
# ==========================================
# HasAns evaluation extraction on SQuADv2 validation
# (aligned with your Google Drive structure)
# ==========================================

import os
import json
import math
import torch
import pandas as pd
import evaluate

from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import StoppingCriteria, StoppingCriteriaList

# -----------------------------
# Base paths (YOUR setup)
# -----------------------------
BASE_RESULTS_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments"
MODEL_DIR = os.path.join(BASE_RESULTS_DIR, "models_Qwen2-1.5B-Instruct_pca")
#ANALYSIS_DIR = os.path.join(BASE_RESULTS_DIR, "analysis_Qwen2-1.5B-Instruct_pca_spike")
ANALYSIS_DIR = os.path.join(BASE_RESULTS_DIR, "analysis_Qwen2-1.5B-Instruct_pca_spike_first_token_hs")
os.makedirs(ANALYSIS_DIR, exist_ok=True)

# -----------------------------
# Model / experiment config
# -----------------------------
MODEL_NAME = "Qwen/Qwen2-1.5B-Instruct"
MODEL_NAME_TAG = MODEL_NAME.replace("/", "-")

# IMPORTANT: use same sparsities as your experiments
#SPARSITIES = [0.0, 0.10, 0.25, 0.40, 0.60]
SPARSITIES = [0.0,0.25,0.60]
# Dataset
EVAL_DATASET_NAME = "squad2"
CALIBRATION_DATASET = "coqa"

# Hidden-state config
ACTIVATION_KIND = "mlp_output"

# Save all transformer layers
NUM_LAYERS = 28
LAYERS_TO_SAVE = list(range(NUM_LAYERS))

print("Layers to save:", LAYERS_TO_SAVE)

# Inference config
BATCH_SIZE = 8
MAX_NEW_TOKENS = 32

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# -----------------------------
# Output directories
# -----------------------------
HASANS_EVAL_DIR = os.path.join(
    ANALYSIS_DIR,
    f"squad2_hasans_eval_all_layers_{ACTIVATION_KIND}"
)
os.makedirs(HASANS_EVAL_DIR, exist_ok=True)

print("HASANS_EVAL_DIR:", HASANS_EVAL_DIR)
print("BASE_RESULTS_DIR:", BASE_RESULTS_DIR)
print("MODEL_DIR:", MODEL_DIR)
print("ANALYSIS_DIR:", ANALYSIS_DIR)
print("HASANS_EVAL_DIR:", HASANS_EVAL_DIR)
print("Sparsities:", SPARSITIES)
print("Activation kind:", ACTIVATION_KIND)

Layers to save: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]
DEVICE: cuda
HASANS_EVAL_DIR: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/analysis_Qwen2-1.5B-Instruct_pca_spike_first_token_hs/squad2_hasans_eval_all_layers_mlp_output
BASE_RESULTS_DIR: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments
MODEL_DIR: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/models_Qwen2-1.5B-Instruct_pca
ANALYSIS_DIR: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/analysis_Qwen2-1.5B-Instruct_pca_spike_first_token_hs
HASANS_EVAL_DIR: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/analysis_Qwen2-1.5B-Instruct_pca_spike_first_token_hs/squad2_hasans_eval_all_layers_mlp_output
Sparsities: [0.0, 0.25, 0.6]
Activation kind: mlp_output


In [ ]:
####################
#From here on custom HasAns

In [4]:
# -----------------------------
# Load SQuAD v2 HasAns subset
# -----------------------------
ds_full = load_dataset("squad_v2", split="validation")
ds_hasans = ds_full.filter(lambda x: len(x["answers"]["text"]) > 0)

print("Full validation size:", len(ds_full))
print("HasAns validation size:", len(ds_hasans))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Full validation size: 11873
HasAns validation size: 5928


In [5]:
# -----------------------------
# Official SQuAD v2 metric
# -----------------------------
squad_v2_metric = evaluate.load("squad_v2")

In [ ]:
def pool_last_nonpad_token(hidden_states, attention_mask):
    """
    hidden_states: (B, T, D)
    attention_mask: (B, T)
    returns: (B, D)
    """
    lengths = attention_mask.sum(dim=1) - 1
    batch_idx = torch.arange(hidden_states.shape[0], device=hidden_states.device)
    return hidden_states[batch_idx, lengths]

In [6]:
class StopOnSubsequence(StoppingCriteria):
    def __init__(self, stop_sequences_ids):
        super().__init__()
        self.stop_sequences_ids = stop_sequences_ids

    def __call__(self, input_ids, scores, **kwargs):
        for stop_ids in self.stop_sequences_ids:
            stop_len = len(stop_ids)
            if stop_len == 0:
                continue
            if input_ids.shape[1] >= stop_len:
                if input_ids[0, -stop_len:].tolist() == stop_ids:
                    return True
        return False


def build_newline_stopping(tokenizer):
    newline_ids = tokenizer.encode("\n", add_special_tokens=False)
    return StoppingCriteriaList([StopOnSubsequence([newline_ids])])


@torch.inference_mode()
def generate_until_newline_batch(model, tokenizer, input_ids, attention_mask, max_new_tokens):
    """
    Approximate lm_eval generate_until(..., until=['\\n'])
    """
    stopping = build_newline_stopping(tokenizer)

    gen_out = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        stopping_criteria=stopping,
    )

    generated_only = gen_out[:, input_ids.shape[1]:]
    decoded = tokenizer.batch_decode(generated_only, skip_special_tokens=True)

    # lm_eval-style: cut at first newline, no extra cleaning
    decoded = [text.split("\n")[0] for text in decoded]
    return gen_out, decoded


@torch.inference_mode()
def loglikelihood_of_continuation_batch(model, tokenizer, prompts, continuation, device):
    """
    Computes log p(continuation | prompt) for each prompt in batch.
    Used to mimic lm_eval's scoring of ' unanswerable'.
    """
    prompt_enc = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    )
    cont_enc = tokenizer(
        [continuation] * len(prompts),
        return_tensors="pt",
        padding=True,
        truncation=True,
        add_special_tokens=False,
    )

    prompt_ids = prompt_enc["input_ids"].to(device)
    prompt_mask = prompt_enc["attention_mask"].to(device)
    cont_ids = cont_enc["input_ids"].to(device)
    cont_mask = cont_enc["attention_mask"].to(device)

    input_ids = torch.cat([prompt_ids, cont_ids], dim=1)
    attention_mask = torch.cat([prompt_mask, cont_mask], dim=1)

    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    logits = outputs.logits[:, :-1, :]
    labels = input_ids[:, 1:]

    log_probs = torch.log_softmax(logits, dim=-1)
    token_log_probs = log_probs.gather(-1, labels.unsqueeze(-1)).squeeze(-1)

    batch_loglikelihoods = []

    for b in range(input_ids.shape[0]):
        prompt_len = int(prompt_mask[b].sum().item())
        cont_len = int(cont_mask[b].sum().item())

        start = prompt_len - 1
        end = start + cont_len

        seq_logprob = token_log_probs[b, start:end].sum().item()
        batch_loglikelihoods.append(seq_logprob)

    return batch_loglikelihoods

In [7]:
@torch.inference_mode()
def generate_until_newline_batch(model, tokenizer, input_ids, attention_mask, max_new_tokens):
    stopping = build_newline_stopping(tokenizer)

    generated = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        num_beams=1,
        stopping_criteria=stopping,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    gen_only = generated[:, input_ids.shape[1]:]
    decoded = tokenizer.batch_decode(gen_only, skip_special_tokens=True)

    return generated, decoded

In [8]:
# LM-EVAL-STYLE:
# We keep the function name, but use official squad_v2 scoring.

def compute_metrics_against_gold_list(prediction, gold_answers, sample_id="dummy"):
    pred = {
        "id": str(sample_id),
        "prediction_text": str(prediction),
        "no_answer_probability": 0.0,
    }
    ref = {
        "id": str(sample_id),
        "answers": {
            "text": list(gold_answers),
            "answer_start": [0] * len(gold_answers),
        },
    }

    out = squad_v2_metric.compute(
        predictions=[pred],
        references=[ref],
    )

    return float(out["f1"]), float(out["exact"])

In [9]:
# =========================
# LM-EVAL-STYLE PROMPTING
# =========================

def build_prompt(example):
    """
    Match lm_eval SQuAD2 prompt format.
    """
    title = str(example.get("title", "")).strip()
    context = str(example["context"]).strip()
    question = str(example["question"]).strip()

    prompt = (
        "Title: " + title + "\n\n"
        + "Background: " + context + "\n\n"
        + "Question: " + question + "\n\n"
        + "Answer:"
    )
    return prompt


def clean_generated_text(text, prompt_text=None):
    """
    Keep same denomination, but make behavior lm_eval-style:
    - remove prompt only if it appears in full
    - stop at first newline
    - no manual stripping of prefixes like 'The answer is'
    """
    text = str(text)

    if prompt_text is not None and text.startswith(prompt_text):
        text = text[len(prompt_text):]

    text = text.split("\n")[0]
    return text

In [8]:
def register_mlp_output_hooks_all_layers(model, layer_indices, storage_dict):
    """
    Register one forward hook per requested layer.
    storage_dict[layer_idx] will receive one tensor per forward call, shape (B, T, D)
    """
    handles = []

    for layer_idx in layer_indices:
        target_module = model.model.layers[layer_idx].mlp

        def make_hook(idx):
            def hook_fn(module, inputs, output):
                out = output[0] if isinstance(output, tuple) else output
                storage_dict[idx].append(out.detach().float().cpu())
            return hook_fn

        handle = target_module.register_forward_hook(make_hook(layer_idx))
        handles.append(handle)

    return handles

In [10]:
def register_mlp_output_hooks_first_generated_all_layers(model, layer_indices, storage_dict):
    """
    Register one forward hook per requested layer.

    For each layer, we store ONLY the hidden state corresponding to the
    first generated token step.

    During generation with KV cache:
      - prompt pass typically has seq_len > 1
      - first generated token step has seq_len == 1
      - later generated token steps also have seq_len == 1

    So we save only the FIRST time we see seq_len == 1 for each layer.
    Stored tensor shape per batch: (B, D)
    """
    handles = []
    captured_once = {layer_idx: False for layer_idx in layer_indices}

    for layer_idx in layer_indices:
        target_module = model.model.layers[layer_idx].mlp

        def make_hook(idx):
            def hook_fn(module, inputs, output):
                out = output[0] if isinstance(output, tuple) else output  # (B, T, D) or similar

                # We only want the first generation step
                if captured_once[idx]:
                    return

                if out.dim() != 3:
                    return

                batch_size, seq_len, hidden_dim = out.shape

                # During generation with cache, first generated token step has seq_len == 1
                if seq_len == 1:
                    storage_dict[idx].append(out[:, 0, :].detach().float().cpu())  # (B, D)
                    captured_once[idx] = True

            return hook_fn

        handle = target_module.register_forward_hook(make_hook(layer_idx))
        handles.append(handle)

    return handles

In [19]:
def get_first_generated_token_index(attention_mask):
    """
    Returns index of first generated token position (i.e. length of prompt)
    """
    return attention_mask.sum(dim=1)

In [ ]:
@torch.inference_mode()
def run_hasans_eval_extraction_for_sparsity(sparsity):
    sliced_path = os.path.join(
        MODEL_DIR,
        f"Qwen-Qwen2-1p5B-Instruct_{CALIBRATION_DATASET}_s{sparsity:.2f}".replace(".", "p")
    )

    print(f"\n[LOAD] calibration={CALIBRATION_DATASET} | sparsity={sparsity} from {sliced_path}")

    model_adapter, tokenizer = hf_utils.load_sliced_model(
        MODEL_NAME,
        sliced_path,
        sparsity=sparsity,
        token=None,
        round_interval=8,
    )

    model = model_adapter.model
    model.eval()
    model.to(DEVICE)

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    tokenizer.padding_side = "left"

    all_hidden_by_layer = {layer_idx: [] for layer_idx in LAYERS_TO_SAVE}
    all_rows = []

    all_predictions = []
    all_references = []

    for start in tqdm(range(0, len(ds_hasans), BATCH_SIZE), desc=f"{CALIBRATION_DATASET} | s={sparsity}"):
        batch = ds_hasans[start:start + BATCH_SIZE]
        batch_size = len(batch["id"])

        prompts = [
            build_prompt({
                "title": batch["title"][i] if "title" in batch else "",
                "context": batch["context"][i],
                "question": batch["question"][i],
            })
            for i in range(batch_size)
        ]

        gold_lists = [batch["answers"][i]["text"] for i in range(batch_size)]
        gold_answer_starts = [batch["answers"][i]["answer_start"] for i in range(batch_size)]
        ids = [batch["id"][i] for i in range(batch_size)]
        titles = [batch["title"][i] if "title" in batch else "" for i in range(batch_size)]
        contexts = [batch["context"][i] for i in range(batch_size)]
        questions = [batch["question"][i] for i in range(batch_size)]

        enc = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
        )

        input_ids = enc["input_ids"].to(DEVICE)
        attention_mask = enc["attention_mask"].to(DEVICE)

        # -------------------------------------------------
        # 1) Capture prompt hidden states for ALL requested layers
        # -------------------------------------------------
        hook_storage = {layer_idx: [] for layer_idx in LAYERS_TO_SAVE}
        hook_handles = register_mlp_output_hooks_all_layers(model, LAYERS_TO_SAVE, hook_storage)

        _ = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        for h in hook_handles:
            h.remove()

        for layer_idx in LAYERS_TO_SAVE:
            if len(hook_storage[layer_idx]) == 0:
                raise RuntimeError(f"Hook did not capture activations for layer {layer_idx}")

            hidden_batch = hook_storage[layer_idx][0]
            pooled = pool_last_nonpad_token(
                hidden_batch.to(DEVICE),
                attention_mask
            ).detach().float().cpu()

            all_hidden_by_layer[layer_idx].append(pooled)

        # -------------------------------------------------
        # 2) LM-EVAL-STYLE generation
        # -------------------------------------------------
        _, decoded = generate_until_newline_batch(
            model=model,
            tokenizer=tokenizer,
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=MAX_NEW_TOKENS,
        )

        # -------------------------------------------------
        # 3) LM-EVAL-STYLE no-answer score
        # -------------------------------------------------
        logprob_unanswerable_batch = loglikelihood_of_continuation_batch(
            model=model,
            tokenizer=tokenizer,
            prompts=prompts,
            continuation=" unanswerable",
            device=DEVICE,
        )

        # -------------------------------------------------
        # 4) Build official squad_v2 predictions/references
        # -------------------------------------------------
        for i in range(batch_size):
            pred = clean_generated_text(decoded[i], prompt_text=None)
            gold_answers = gold_lists[i]
            gold_starts = gold_answer_starts[i]

            no_answer_probability = math.exp(logprob_unanswerable_batch[i])

            pred_dict = {
                "id": ids[i],
                "prediction_text": pred,
                "no_answer_probability": no_answer_probability,
            }
            ref_dict = {
                "id": ids[i],
                "answers": {
                    "text": gold_answers,
                    "answer_start": gold_starts,
                },
            }

            all_predictions.append(pred_dict)
            all_references.append(ref_dict)

            sample_metric = squad_v2_metric.compute(
                predictions=[pred_dict],
                references=[ref_dict],
            )

            all_rows.append({
                "sample_id": ids[i],
                "row_idx": len(all_rows),
                "title": titles[i],
                "question": questions[i],
                "context": contexts[i],
                "prompt": prompts[i],
                "prediction_raw": decoded[i],
                "prediction": pred,
                "gold_answers": json.dumps(gold_answers, ensure_ascii=False),
                "no_answer_probability": float(no_answer_probability),
                "f1": float(sample_metric["f1"]),
                "exact": float(sample_metric["exact"]),
            })

    hidden_by_layer = {}
    for layer_idx in LAYERS_TO_SAVE:
        hidden_by_layer[layer_idx] = torch.cat(all_hidden_by_layer[layer_idx], dim=0)
        print(f"Layer {layer_idx} hidden-state matrix: {tuple(hidden_by_layer[layer_idx].shape)}")

    df_rows = pd.DataFrame(all_rows)

    overall_metrics = squad_v2_metric.compute(
        predictions=all_predictions,
        references=all_references,
    )

    print("Rows:", len(all_rows))
    print(f"[OFFICIAL] Exact={overall_metrics['exact']:.4f} | F1={overall_metrics['f1']:.4f}")

    del model_adapter
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return hidden_by_layer, df_rows

In [11]:
@torch.inference_mode()
def run_hasans_eval_extraction_for_sparsity(sparsity):
    sliced_path = os.path.join(
        MODEL_DIR,
        f"Qwen-Qwen2-1p5B-Instruct_{CALIBRATION_DATASET}_s{sparsity:.2f}".replace(".", "p")
    )

    print(f"\n[LOAD] calibration={CALIBRATION_DATASET} | sparsity={sparsity} from {sliced_path}")

    model_adapter, tokenizer = hf_utils.load_sliced_model(
        MODEL_NAME,
        sliced_path,
        sparsity=sparsity,
        token=None,
        round_interval=8,
    )

    model = model_adapter.model
    model.eval()
    model.to(DEVICE)

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    tokenizer.padding_side = "left"

    all_hidden_by_layer = {layer_idx: [] for layer_idx in LAYERS_TO_SAVE}
    all_rows = []

    all_predictions = []
    all_references = []

    for start in tqdm(range(0, len(ds_hasans), BATCH_SIZE), desc=f"{CALIBRATION_DATASET} | s={sparsity}"):
        batch = ds_hasans[start:start + BATCH_SIZE]
        batch_size = len(batch["id"])

        prompts = [
            build_prompt({
                "title": batch["title"][i] if "title" in batch else "",
                "context": batch["context"][i],
                "question": batch["question"][i],
            })
            for i in range(batch_size)
        ]

        gold_lists = [batch["answers"][i]["text"] for i in range(batch_size)]
        gold_answer_starts = [batch["answers"][i]["answer_start"] for i in range(batch_size)]
        ids = [batch["id"][i] for i in range(batch_size)]
        titles = [batch["title"][i] if "title" in batch else "" for i in range(batch_size)]
        contexts = [batch["context"][i] for i in range(batch_size)]
        questions = [batch["question"][i] for i in range(batch_size)]

        enc = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
        )

        input_ids = enc["input_ids"].to(DEVICE)
        attention_mask = enc["attention_mask"].to(DEVICE)

        # -------------------------------------------------
        # 1) Capture FIRST GENERATED TOKEN hidden states
        # -------------------------------------------------
        hook_storage = {layer_idx: [] for layer_idx in LAYERS_TO_SAVE}
        hook_handles = register_mlp_output_hooks_first_generated_all_layers(
            model, LAYERS_TO_SAVE, hook_storage
        )

        # -------------------------------------------------
        # 2) Generation
        # -------------------------------------------------
        _, decoded = generate_until_newline_batch(
            model=model,
            tokenizer=tokenizer,
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=MAX_NEW_TOKENS,
        )

        for h in hook_handles:
            h.remove()

        # Store first generated token HS for all requested layers
        for layer_idx in LAYERS_TO_SAVE:
            if len(hook_storage[layer_idx]) == 0:
                raise RuntimeError(
                    f"Hook did not capture first generated-token activations for layer {layer_idx}"
                )

            first_gen_hs = hook_storage[layer_idx][0]   # (B, D)
            all_hidden_by_layer[layer_idx].append(first_gen_hs)

        # -------------------------------------------------
        # 3) LM-EVAL-STYLE no-answer score
        # -------------------------------------------------
        logprob_unanswerable_batch = loglikelihood_of_continuation_batch(
            model=model,
            tokenizer=tokenizer,
            prompts=prompts,
            continuation=" unanswerable",
            device=DEVICE,
        )

        # -------------------------------------------------
        # 4) Build official squad_v2 predictions/references
        # -------------------------------------------------
        for i in range(batch_size):
            pred = clean_generated_text(decoded[i], prompt_text=None)
            gold_answers = gold_lists[i]
            gold_starts = gold_answer_starts[i]

            no_answer_probability = math.exp(logprob_unanswerable_batch[i])

            pred_dict = {
                "id": ids[i],
                "prediction_text": pred,
                "no_answer_probability": no_answer_probability,
            }
            ref_dict = {
                "id": ids[i],
                "answers": {
                    "text": gold_answers,
                    "answer_start": gold_starts,
                },
            }

            all_predictions.append(pred_dict)
            all_references.append(ref_dict)

            sample_metric = squad_v2_metric.compute(
                predictions=[pred_dict],
                references=[ref_dict],
            )

            all_rows.append({
                "sample_id": ids[i],
                "row_idx": len(all_rows),
                "title": titles[i],
                "question": questions[i],
                "context": contexts[i],
                "prompt": prompts[i],
                "prediction_raw": decoded[i],
                "prediction": pred,
                "gold_answers": json.dumps(gold_answers, ensure_ascii=False),
                "no_answer_probability": float(no_answer_probability),
                "f1": float(sample_metric["f1"]),
                "exact": float(sample_metric["exact"]),
            })

    hidden_by_layer = {}
    for layer_idx in LAYERS_TO_SAVE:
        hidden_by_layer[layer_idx] = torch.cat(all_hidden_by_layer[layer_idx], dim=0)
        print(f"Layer {layer_idx} first-generated-token hidden-state matrix: {tuple(hidden_by_layer[layer_idx].shape)}")

    df_rows = pd.DataFrame(all_rows)

    overall_metrics = squad_v2_metric.compute(
        predictions=all_predictions,
        references=all_references,
    )

    print("Rows:", len(all_rows))
    print(f"[OFFICIAL] Exact={overall_metrics['exact']:.4f} | F1={overall_metrics['f1']:.4f}")

    del model_adapter
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return hidden_by_layer, df_rows

In [ ]:
for sparsity in [0.25,0.6]:
    hidden_by_layer, df_metrics = run_hasans_eval_extraction_for_sparsity(sparsity)

    save_subdir = os.path.join(
        MODEL_DIR,
        f"Qwen-Qwen2-1p5B-Instruct_{CALIBRATION_DATASET}_s{sparsity:.2f}".replace(".", "p"),
        "hidden_states"
    )
    os.makedirs(save_subdir, exist_ok=True)

    # save one file per layer
    for layer_idx, H in hidden_by_layer.items():
        hs_path = os.path.join(
            save_subdir,
            f"layer_{layer_idx}_{ACTIVATION_KIND}_hasans_eval_first_token_hs.pt"
        )
        torch.save(H, hs_path)
        print(f"[SAVED] hidden states -> {hs_path}")

    metrics_path = os.path.join(
        HASANS_EVAL_DIR,
        f"{MODEL_NAME_TAG}_{CALIBRATION_DATASET}_on_{EVAL_DATASET_NAME}_hasans_s{sparsity:.2f}".replace(".", "p")
        + "_sample_metrics.csv"
    )

    df_metrics.to_csv(metrics_path, index=False)

    print(f"[SAVED] metrics       -> {metrics_path}")
    print(f"[MEAN SAMPLE] F1={df_metrics['f1'].mean():.4f} | EM={df_metrics['exact'].mean():.4f}")

In [16]:
summary_rows = []

for sparsity in SPARSITIES:
    metrics_path = os.path.join(
        HASANS_EVAL_DIR,
        f"{MODEL_NAME_TAG}_{CALIBRATION_DATASET}_on_{EVAL_DATASET_NAME}_hasans_s{sparsity:.2f}".replace(".", "p")
        + "_sample_metrics.csv"
    )

    df = pd.read_csv(metrics_path)

    summary_rows.append({
        "sparsity": sparsity,
        "num_samples": len(df),
        "mean_f1": df["f1"].mean(),
        "mean_em": df["exact"].mean(),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

,sparsity,num_samples,mean_f1,mean_em
0,0.00,5928,39.029010,17.139001
1,0.25,5928,55.522876,33.620108
2,0.60,5928,2.627478,0.000000


In [ ]:
@torch.inference_mode()
def sanity_check_prompt_effect(sparsity=0.0, num_samples=50):
    sliced_path = os.path.join(
        MODEL_DIR,
        f"Qwen-Qwen2-1p5B-Instruct_{CALIBRATION_DATASET}_s{sparsity:.2f}".replace(".", "p")
    )

    print(f"[LOAD] calibration={CALIBRATION_DATASET} | sparsity={sparsity} from {sliced_path}")

    model_adapter, tokenizer = hf_utils.load_sliced_model(
        MODEL_NAME,
        sliced_path,
        sparsity=sparsity,
        token=None,
        round_interval=8,
    )
    model = model_adapter.model
    model.eval()
    model.to(DEVICE)

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    tokenizer.padding_side = "left"
    rows = []

    subset = ds_hasans.select(range(min(num_samples, len(ds_hasans))))

    for i in tqdm(range(len(subset)), desc="sanity-check"):
        ex = subset[i]

        old_prompt = ex["context"] + "\n\n" + ex["question"]
        new_prompt = build_prompt(ex)

        for prompt_name, prompt in [("old", old_prompt), ("new", new_prompt)]:
            enc = tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True
            ).to(DEVICE)

            _, decoded_list = generate_until_newline_batch(
                model=model,
                tokenizer=tokenizer,
                input_ids=enc["input_ids"],
                attention_mask=enc["attention_mask"],
                max_new_tokens=MAX_NEW_TOKENS,
            )

            decoded = decoded_list[0]
            pred = clean_generated_text(decoded)

            gold_answers = ex["answers"]["text"]
            gold_starts = ex["answers"]["answer_start"]

            no_answer_logprob = loglikelihood_of_continuation_batch(
                model=model,
                tokenizer=tokenizer,
                prompts=[prompt],
                continuation=" unanswerable",
                device=DEVICE,
            )[0]

            no_answer_probability = math.exp(no_answer_logprob)

            pred_dict = {
                "id": ex["id"],
                "prediction_text": pred,
                "no_answer_probability": no_answer_probability,
            }
            ref_dict = {
                "id": ex["id"],
                "answers": {
                    "text": gold_answers,
                    "answer_start": gold_starts,
                },
            }

            metric = squad_v2_metric.compute(
                predictions=[pred_dict],
                references=[ref_dict],
            )

            rows.append({
                "sample_id": ex["id"],
                "prompt_type": prompt_name,
                "question": ex["question"],
                "gold_answers": json.dumps(gold_answers, ensure_ascii=False),
                "prediction_raw": decoded,
                "prediction": pred,
                "no_answer_probability": float(no_answer_probability),
                "f1": float(metric["f1"]),
                "exact": float(metric["exact"]),
            })

    del model_adapter
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    df = pd.DataFrame(rows)
    summary = df.groupby("prompt_type")[["f1", "exact"]].mean().reset_index()
    return df, summary

In [ ]:
sanity_df, sanity_summary = sanity_check_prompt_effect(sparsity=0.0, num_samples=50)

display(sanity_summary)

print("\nExamples where old/new differ:\n")
for sample_id in sanity_df["sample_id"].unique()[:5]:
    cur = sanity_df[sanity_df["sample_id"] == sample_id]
    print("=" * 120)
    for _, row in cur.iterrows():
        print(f"[{row['prompt_type']}]")
        print("Q:", row["question"])
        print("Gold:", row["gold_answers"])
        print("Pred:", row["prediction"])
        print("F1:", row["f1"], "| EM:", row["exact"])
        print()

In [ ]:
sanity_df, sanity_summary = sanity_check_prompt_effect(sparsity=0.25, num_samples=50)

display(sanity_summary)

print("\nExamples where old/new differ:\n")
for sample_id in sanity_df["sample_id"].unique()[:5]:
    cur = sanity_df[sanity_df["sample_id"] == sample_id]
    print("=" * 120)
    for _, row in cur.iterrows():
        print(f"[{row['prompt_type']}]")
        print("Q:", row["question"])
        print("Gold:", row["gold_answers"])
        print("Pred:", row["prediction"])
        print("F1:", row["f1"], "| EM:", row["exact"])
        print()

In [ ]:
##################
###Let's start now once more with the analysis

In [17]:
import os
import ast
import pandas as pd
import numpy as np
from IPython.display import display

# =========================
# CONFIG
# =========================
BASE_DIR = "/content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/analysis_Qwen2-1.5B-Instruct_pca_spike_first_token_hs/squad2_hasans_eval_all_layers_mlp_output"

CSV_PATHS = {
    "0.0": os.path.join(BASE_DIR, "Qwen-Qwen2-1p5B-Instruct_coqa_on_squad2_hasans_s0p00_sample_metrics.csv"),
    "0.25": os.path.join(BASE_DIR, "Qwen-Qwen2-1p5B-Instruct_coqa_on_squad2_hasans_s0p25_sample_metrics.csv"),
    "0.6": os.path.join(BASE_DIR, "Qwen-Qwen2-1p5B-Instruct_coqa_on_squad2_hasans_s0p60_sample_metrics.csv"),
}

# correctness threshold for F1-based transition analysis
F1_THRESHOLD = 50.0

In [18]:
dfs = {}

for s, path in CSV_PATHS.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing file for sparsity {s}: {path}")

    df = pd.read_csv(path)
    dfs[s] = df

    print(f"\n=== Sparsity {s} ===")
    print("Path:", path)
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))


=== Sparsity 0.0 ===
Path: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/analysis_Qwen2-1.5B-Instruct_pca_spike_first_token_hs/squad2_hasans_eval_all_layers_mlp_output/Qwen-Qwen2-1p5B-Instruct_coqa_on_squad2_hasans_s0p00_sample_metrics.csv
Shape: (5928, 12)
Columns: ['sample_id', 'row_idx', 'title', 'question', 'context', 'prompt', 'prediction_raw', 'prediction', 'gold_answers', 'no_answer_probability', 'f1', 'exact']

=== Sparsity 0.25 ===
Path: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/analysis_Qwen2-1.5B-Instruct_pca_spike_first_token_hs/squad2_hasans_eval_all_layers_mlp_output/Qwen-Qwen2-1p5B-Instruct_coqa_on_squad2_hasans_s0p25_sample_metrics.csv
Shape: (5928, 12)
Columns: ['sample_id', 'row_idx', 'title', 'question', 'context', 'prompt', 'prediction_raw', 'prediction', 'gold_answers', 'no_answer_probability', 'f1', 'exact']

=== Sparsity 0.6 ===
Path: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/analysis_Qwen2-1.5B-Instruct_pca_spike_first_toke

In [19]:
def parse_gold_answers(x):
    if pd.isna(x):
        return []

    if isinstance(x, list):
        return x

    if isinstance(x, str):
        x = x.strip()
        if not x:
            return []
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return parsed
            return [str(parsed)]
        except Exception:
            return [x]

    return [str(x)]


required_cols = ["sample_id", "prediction", "f1", "exact"]

for s, df in dfs.items():
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Sparsity {s} missing required columns: {missing}")

# sort by sample_id for alignment
for s in dfs:
    dfs[s] = dfs[s].sort_values("sample_id").reset_index(drop=True)

# verify same sample_id order across all chosen sparsities
sparsity_keys = list(CSV_PATHS.keys())
base_key = sparsity_keys[0]
base_ids = dfs[base_key]["sample_id"].tolist()

for s in sparsity_keys[1:]:
    ids = dfs[s]["sample_id"].tolist()
    if ids != base_ids:
        raise ValueError(f"sample_id mismatch between {base_key} and {s}")

# parse optional gold_answers
for s in dfs:
    if "gold_answers" in dfs[s].columns:
        dfs[s]["gold_answers_parsed"] = dfs[s]["gold_answers"].apply(parse_gold_answers)
    else:
        dfs[s]["gold_answers_parsed"] = [[] for _ in range(len(dfs[s]))]

print("All sample_id alignments are correct.")
print("Number of samples:", len(base_ids))

All sample_id alignments are correct.
Number of samples: 5928


In [20]:
keep_cols_optional = [
    "question",
    "context",
    "gold_answers",
    "gold_answers_parsed",
]

# base info from the first sparsity
base_key = list(CSV_PATHS.keys())[0]
base_df = dfs[base_key][["sample_id"]].copy()

for col in keep_cols_optional:
    if col in dfs[base_key].columns:
        base_df[col] = dfs[base_key][col]

for s, df in dfs.items():
    tmp = df[["sample_id", "prediction", "f1", "exact"]].copy()
    tmp = tmp.rename(columns={
        "prediction": f"prediction_s{s}",
        "f1": f"f1_s{s}",
        "exact": f"exact_s{s}",
    })
    base_df = base_df.merge(tmp, on="sample_id", how="left")

display(base_df.head())
print("Merged shape:", base_df.shape)

,sample_id,question,context,gold_answers,gold_answers_parsed,prediction_s0.0,f1_s0.0,exact_s0.0,prediction_s0.25,f1_s0.25,exact_s0.25,prediction_s0.6,f1_s0.6,exact_s0.6
0,56ddde6b9a695914005b9628,In what country is Normandy located?,The Normans (Norman: Nourmands; French: Norman...,"[""France"", ""France"", ""France"", ""France""]","[France, France, France, France]",Normandy is located in France. It is a region...,9.090909,0.0,France.,100.000000,100.0,In what country is the French?,0.000000,0.0
1,56ddde6b9a695914005b9629,When were the Normans in Normandy?,The Normans (Norman: Nourmands; French: Norman...,"[""10th and 11th centuries"", ""in the 10th and 1...","[10th and 11th centuries, in the 10th and 11th...",The Normans were in Normandy from the 10th ce...,19.047619,0.0,The Normans were in Normandy in the 10th and ...,71.428571,0.0,What was the name of the people in the 10th c...,30.769231,0.0
2,56ddde6b9a695914005b962a,From which countries did the Norse originate?,The Normans (Norman: Nourmands; French: Norman...,"[""Denmark, Iceland and Norway"", ""Denmark, Icel...","[Denmark, Iceland and Norway, Denmark, Iceland...",The Norse (also known as Normans) originated ...,28.571429,0.0,Norse.,0.000000,0.0,The Norse are the Norse. The Norse are the No...,0.000000,0.0
3,56ddde6b9a695914005b962b,Who was the Norse leader?,The Normans (Norman: Nourmands; French: Norman...,"[""Rollo"", ""Rollo"", ""Rollo"", ""Rollo""]","[Rollo, Rollo, Rollo, Rollo]",Rollo,100.000000,100.0,Rollo.,100.000000,100.0,Who was the Norse leader?,0.000000,0.0
4,56ddde6b9a695914005b962c,What century did the Normans first gain their ...,The Normans (Norman: Nourmands; French: Norman...,"[""10th century"", ""the first half of the 10th c...","[10th century, the first half of the 10th cent...",The Normans first gained their separate ident...,35.714286,0.0,The 10th century.,100.000000,100.0,What century did the Normans first gain their...,28.571429,0.0


Merged shape: (5928, 14)


In [21]:
df = base_df.copy()

ordered_sparsities = list(CSV_PATHS.keys())

# binary correctness from exact
for s in ordered_sparsities:
    df[f"exact_correct_s{s}"] = (df[f"exact_s{s}"] == 100).astype(int)

# binary correctness from F1 threshold
for s in ordered_sparsities:
    df[f"f1_correct_s{s}"] = (df[f"f1_s{s}"] >= F1_THRESHOLD).astype(int)

# transition strings
df["transition_exact"] = "->".join([f"{{exact_correct_s{s}}}" for s in ordered_sparsities])
df["transition_exact"] = df.apply(
    lambda row: "->".join(str(row[f"exact_correct_s{s}"]) for s in ordered_sparsities),
    axis=1
)

df["transition_f1"] = df.apply(
    lambda row: "->".join(str(row[f"f1_correct_s{s}"]) for s in ordered_sparsities),
    axis=1
)

display_cols = ["sample_id"]
for s in ordered_sparsities:
    display_cols += [f"exact_s{s}"]
for s in ordered_sparsities:
    display_cols += [f"f1_s{s}"]
display_cols += ["transition_exact", "transition_f1"]

display(df[display_cols].head())

,sample_id,exact_s0.0,exact_s0.25,exact_s0.6,f1_s0.0,f1_s0.25,f1_s0.6,transition_exact,transition_f1
0,56ddde6b9a695914005b9628,0.0,100.0,0.0,9.090909,100.000000,0.000000,0->1->0,0->1->0
1,56ddde6b9a695914005b9629,0.0,0.0,0.0,19.047619,71.428571,30.769231,0->0->0,0->1->0
2,56ddde6b9a695914005b962a,0.0,0.0,0.0,28.571429,0.000000,0.000000,0->0->0,0->0->0
3,56ddde6b9a695914005b962b,100.0,100.0,0.0,100.000000,100.000000,0.000000,1->1->0,1->1->0
4,56ddde6b9a695914005b962c,0.0,100.0,0.0,35.714286,100.000000,28.571429,0->1->0,0->1->0


In [22]:
######this was with exact being 1 instead of 100 and f1 being 0.5 instead of 50
def transition_table(df, transition_col):
    out = df[transition_col].value_counts().sort_index().reset_index()
    out.columns = ["transition", "count"]
    out["percentage"] = 100.0 * out["count"] / out["count"].sum()
    return out.sort_values("count", ascending=False).reset_index(drop=True)


print("=== EXACT transitions ===")
display(transition_table(df, "transition_exact"))

print("=== F1>=0.5 transitions ===")
display(transition_table(df, "transition_f1"))

=== EXACT transitions ===


,transition,count,percentage
0,0->0->0,3580,60.391363
1,0->1->0,1332,22.469636
2,1->1->0,661,11.150472
3,1->0->0,355,5.988529


=== F1>=0.5 transitions ===


,transition,count,percentage
0,0->0->0,2106,35.526316
1,0->1->0,1965,33.147773
2,1->1->0,1359,22.925101
3,1->0->0,479,8.080297
4,0->1->1,8,0.134953
5,0->0->1,5,0.084345
6,1->1->1,5,0.084345
7,1->0->1,1,0.016869


In [23]:
def transition_table(df, transition_col):
    out = df[transition_col].value_counts().sort_index().reset_index()
    out.columns = ["transition", "count"]
    out["percentage"] = 100.0 * out["count"] / out["count"].sum()
    return out.sort_values("count", ascending=False).reset_index(drop=True)


print("=== EXACT transitions ===")
display(transition_table(df, "transition_exact"))

print("=== F1>=0.5 transitions ===")
display(transition_table(df, "transition_f1"))

=== EXACT transitions ===


,transition,count,percentage
0,0->0->0,3580,60.391363
1,0->1->0,1332,22.469636
2,1->1->0,661,11.150472
3,1->0->0,355,5.988529


=== F1>=0.5 transitions ===


,transition,count,percentage
0,0->0->0,2106,35.526316
1,0->1->0,1965,33.147773
2,1->1->0,1359,22.925101
3,1->0->0,479,8.080297
4,0->1->1,8,0.134953
5,0->0->1,5,0.084345
6,1->1->1,5,0.084345
7,1->0->1,1,0.016869


In [ ]:
##########old summary

In [24]:
def summarize_by_transition(df, transition_col):
    agg_dict = {"sample_id": ["count"]}

    for s in ordered_sparsities:
        agg_dict[f"f1_s{s}"] = ["mean", "median"]
        agg_dict[f"exact_s{s}"] = ["mean"]

    grouped = df.groupby(transition_col).agg(agg_dict)
    grouped.columns = ["_".join(c).strip("_") for c in grouped.columns]
    grouped = grouped.rename(columns={"sample_id_count": "count"})
    return grouped.sort_values("count", ascending=False).reset_index()


print("=== Summary by EXACT transition ===")
display(summarize_by_transition(df, "transition_exact"))

print("=== Summary by F1 transition ===")
display(summarize_by_transition(df, "transition_f1"))

=== Summary by EXACT transition ===


,transition_exact,count,f1_s0.0_mean,f1_s0.0_median,exact_s0.0_mean,f1_s0.25_mean,f1_s0.25_median,exact_s0.25_mean,f1_s0.6_mean,f1_s0.6_median,exact_s0.6_mean
0,0->0->0,3580,27.099896,21.052632,0.0,33.379459,28.571429,0.0,2.845462,0.0,0.0
1,0->1->0,1332,24.584342,19.047619,0.0,100.000000,100.000000,100.0,2.715796,0.0,0.0
2,1->1->0,661,100.000000,100.000000,100.0,100.000000,100.000000,100.0,1.683954,0.0,0.0
3,1->0->0,355,100.000000,100.000000,100.0,29.129990,26.666667,0.0,1.854647,0.0,0.0


=== Summary by F1 transition ===


,transition_f1,count,f1_s0.0_mean,f1_s0.0_median,exact_s0.0_mean,f1_s0.25_mean,f1_s0.25_median,exact_s0.25_mean,f1_s0.6_mean,f1_s0.6_median,exact_s0.6_mean
0,0->0->0,2106,16.712977,15.384615,0.000000,15.538407,14.285714,0.000000,2.031343,0.000000,0.0
1,0->1->0,1965,20.635533,19.047619,0.000000,85.644285,100.000000,58.727735,2.567740,0.000000,0.0
2,1->1->0,1359,84.823568,100.000000,56.659308,87.796894,100.000000,61.295070,2.715922,0.000000,0.0
3,1->0->0,479,82.542025,100.000000,50.939457,15.759936,11.764706,0.000000,3.115182,0.000000,0.0
4,0->1->1,8,26.259926,34.814815,0.000000,77.391098,71.875000,25.000000,59.346591,57.272727,0.0
5,0->0->1,5,27.507792,24.000000,0.000000,33.117460,40.000000,0.000000,55.428571,57.142857,0.0
6,1->1->1,5,78.095238,66.666667,40.000000,95.000000,100.000000,80.000000,53.409091,50.000000,0.0
7,1->0->1,1,66.666667,66.666667,0.000000,0.000000,0.000000,0.000000,50.000000,50.000000,0.0


In [ ]:
#########################new analysis more deep

In [25]:
import re
import string
from collections import Counter

def simple_normalize_text(s):
    if s is None:
        return ""
    s = str(s).lower().strip()
    s = re.sub(r"\s+", " ", s)
    return s

def simple_tokenize(s):
    s = simple_normalize_text(s)
    s = s.translate(str.maketrans("", "", string.punctuation))
    return [tok for tok in s.split() if tok]

def best_gold_token_overlap_precision(prediction, gold_answers):
    """
    Precision-like overlap:
    shared_tokens / pred_tokens
    We take the best score across all gold answers.
    Range: [0, 1]
    """
    pred_tokens = simple_tokenize(prediction)
    if len(pred_tokens) == 0:
        return 0.0

    best_score = 0.0
    pred_counter = Counter(pred_tokens)

    for gold in gold_answers:
        gold_tokens = simple_tokenize(gold)
        gold_counter = Counter(gold_tokens)
        common = pred_counter & gold_counter
        num_same = sum(common.values())
        score = num_same / max(len(pred_tokens), 1)
        best_score = max(best_score, score)

    return float(best_score)

def is_prediction_substring_of_context(prediction, context):
    """
    1 if normalized prediction appears in normalized context, else 0.
    Empty predictions count as 0.
    """
    pred = simple_normalize_text(prediction)
    ctx = simple_normalize_text(context)

    if pred == "" or ctx == "":
        return 0

    return int(pred in ctx)

def token_jaccard_similarity(a, b):
    """
    Token-set Jaccard similarity between two predictions.
    Range: [0, 1]
    """
    ta = set(simple_tokenize(a))
    tb = set(simple_tokenize(b))

    if len(ta) == 0 and len(tb) == 0:
        return 1.0
    if len(ta | tb) == 0:
        return 0.0

    return float(len(ta & tb) / len(ta | tb))

def exact_string_match(a, b):
    return int(simple_normalize_text(a) == simple_normalize_text(b))

In [26]:
for s in ordered_sparsities:
    pred_col = f"prediction_s{s}"

    df[f"pred_len_chars_s{s}"] = df[pred_col].fillna("").astype(str).apply(len)
    df[f"pred_len_words_s{s}"] = df[pred_col].fillna("").astype(str).apply(
        lambda x: len(simple_tokenize(x))
    )

    df[f"gold_overlap_precision_s{s}"] = df.apply(
        lambda row: best_gold_token_overlap_precision(
            row[pred_col],
            row["gold_answers_parsed"] if "gold_answers_parsed" in row else []
        ),
        axis=1
    )

    if "context" in df.columns:
        df[f"is_substring_context_s{s}"] = df.apply(
            lambda row: is_prediction_substring_of_context(
                row[pred_col],
                row["context"]
            ),
            axis=1
        )
    else:
        df[f"is_substring_context_s{s}"] = 0

    df[f"is_empty_pred_s{s}"] = (df[f"pred_len_words_s{s}"] == 0).astype(int)
    df[f"is_very_short_pred_s{s}"] = (df[f"pred_len_words_s{s}"] <= 2).astype(int)

display_cols = ["sample_id"]
for s in ordered_sparsities:
    display_cols += [
        f"pred_len_words_s{s}",
        f"gold_overlap_precision_s{s}",
        f"is_substring_context_s{s}",
        f"is_empty_pred_s{s}",
        f"is_very_short_pred_s{s}",
    ]

display(df[display_cols].head())

,sample_id,pred_len_words_s0.0,gold_overlap_precision_s0.0,is_substring_context_s0.0,is_empty_pred_s0.0,is_very_short_pred_s0.0,pred_len_words_s0.25,gold_overlap_precision_s0.25,is_substring_context_s0.25,is_empty_pred_s0.25,is_very_short_pred_s0.25,pred_len_words_s0.6,gold_overlap_precision_s0.6,is_substring_context_s0.6,is_empty_pred_s0.6,is_very_short_pred_s0.6
0,56ddde6b9a695914005b9628,28,0.035714,0,0,0,1,1.000000,1,0,1,6,0.000000,0,0,0
1,56ddde6b9a695914005b9629,20,0.150000,0,0,0,11,0.545455,0,0,0,11,0.272727,0,0,0
2,56ddde6b9a695914005b962a,26,0.153846,0,0,0,1,0.000000,0,0,1,27,0.000000,0,0,0
3,56ddde6b9a695914005b962b,1,1.000000,1,0,1,1,1.000000,0,0,1,5,0.000000,0,0,0
4,56ddde6b9a695914005b962c,27,0.259259,0,0,0,3,1.000000,0,0,0,10,0.300000,0,0,0


In [27]:
# prediction similarity across sparsities
# here for your current setup: 0.0 -> 0.25 and 0.25 -> 0.6

pair_list = list(zip(ordered_sparsities[:-1], ordered_sparsities[1:]))

for s1, s2 in pair_list:
    p1 = f"prediction_s{s1}"
    p2 = f"prediction_s{s2}"

    df[f"pred_jaccard_s{s1}_to_s{s2}"] = df.apply(
        lambda row: token_jaccard_similarity(row[p1], row[p2]),
        axis=1
    )

    df[f"pred_exact_match_s{s1}_to_s{s2}"] = df.apply(
        lambda row: exact_string_match(row[p1], row[p2]),
        axis=1
    )

sim_cols = ["sample_id"]
for s1, s2 in pair_list:
    sim_cols += [
        f"pred_jaccard_s{s1}_to_s{s2}",
        f"pred_exact_match_s{s1}_to_s{s2}",
    ]

display(df[sim_cols].head())

,sample_id,pred_jaccard_s0.0_to_s0.25,pred_exact_match_s0.0_to_s0.25,pred_jaccard_s0.25_to_s0.6,pred_exact_match_s0.25_to_s0.6
0,56ddde6b9a695914005b9628,0.052632,0,0.000000,0
1,56ddde6b9a695914005b9629,0.428571,0,0.200000,0
2,56ddde6b9a695914005b962a,0.043478,0,0.333333,0
3,56ddde6b9a695914005b962b,1.000000,0,0.000000,0
4,56ddde6b9a695914005b962c,0.136364,0,0.181818,0


In [28]:
def summarize_by_transition_extended(df, transition_col):
    agg_dict = {"sample_id": ["count"]}

    # performance
    for s in ordered_sparsities:
        agg_dict[f"f1_s{s}"] = ["mean", "median"]
        agg_dict[f"exact_s{s}"] = ["mean"]

    # new behavioral features
    for s in ordered_sparsities:
        agg_dict[f"pred_len_words_s{s}"] = ["mean", "median"]
        agg_dict[f"gold_overlap_precision_s{s}"] = ["mean", "median"]
        agg_dict[f"is_substring_context_s{s}"] = ["mean"]
        agg_dict[f"is_empty_pred_s{s}"] = ["mean"]
        agg_dict[f"is_very_short_pred_s{s}"] = ["mean"]

    # similarity between consecutive sparsities
    for s1, s2 in pair_list:
        agg_dict[f"pred_jaccard_s{s1}_to_s{s2}"] = ["mean", "median"]
        agg_dict[f"pred_exact_match_s{s1}_to_s{s2}"] = ["mean"]

    grouped = df.groupby(transition_col).agg(agg_dict)
    grouped.columns = ["_".join(c).strip("_") for c in grouped.columns]
    grouped = grouped.rename(columns={"sample_id_count": "count"})

    # optional: percentages for binary means
    for s in ordered_sparsities:
        grouped[f"substring_rate_pct_s{s}"] = 100.0 * grouped[f"is_substring_context_s{s}_mean"]
        grouped[f"empty_rate_pct_s{s}"] = 100.0 * grouped[f"is_empty_pred_s{s}_mean"]
        grouped[f"very_short_rate_pct_s{s}"] = 100.0 * grouped[f"is_very_short_pred_s{s}_mean"]

    for s1, s2 in pair_list:
        grouped[f"pred_exact_match_rate_pct_s{s1}_to_s{s2}"] = (
            100.0 * grouped[f"pred_exact_match_s{s1}_to_s{s2}_mean"]
        )

    return grouped.sort_values("count", ascending=False).reset_index()

In [29]:
print("=== Extended summary by EXACT transition ===")
summary_exact_ext = summarize_by_transition_extended(df, "transition_exact")
display(summary_exact_ext)

print("=== Extended summary by F1 transition ===")
summary_f1_ext = summarize_by_transition_extended(df, "transition_f1")
display(summary_f1_ext)

=== Extended summary by EXACT transition ===


,transition_exact,count,f1_s0.0_mean,f1_s0.0_median,exact_s0.0_mean,f1_s0.25_mean,f1_s0.25_median,exact_s0.25_mean,f1_s0.6_mean,f1_s0.6_median,...,empty_rate_pct_s0.0,very_short_rate_pct_s0.0,substring_rate_pct_s0.25,empty_rate_pct_s0.25,very_short_rate_pct_s0.25,substring_rate_pct_s0.6,empty_rate_pct_s0.6,very_short_rate_pct_s0.6,pred_exact_match_rate_pct_s0.0_to_s0.25,pred_exact_match_rate_pct_s0.25_to_s0.6
0,0->0->0,3580,27.099896,21.052632,0.0,33.379459,28.571429,0.0,2.845462,0.0,...,0.0,4.469274,12.122905,1.592179,19.189944,0.139665,0.027933,2.374302,5.391061,0.0
1,0->1->0,1332,24.584342,19.047619,0.0,100.000000,100.000000,100.0,2.715796,0.0,...,0.0,4.204204,24.024024,0.000000,61.261261,0.150150,0.000000,2.027027,0.000000,0.0
2,1->1->0,661,100.000000,100.000000,100.0,100.000000,100.000000,100.0,1.683954,0.0,...,0.0,71.104387,19.667171,0.000000,70.650530,0.151286,0.000000,2.269289,23.146747,0.0
3,1->0->0,355,100.000000,100.000000,100.0,29.129990,26.666667,0.0,1.854647,0.0,...,0.0,62.253521,11.830986,3.098592,30.985915,0.000000,0.000000,2.535211,0.000000,0.0


=== Extended summary by F1 transition ===


,transition_f1,count,f1_s0.0_mean,f1_s0.0_median,exact_s0.0_mean,f1_s0.25_mean,f1_s0.25_median,exact_s0.25_mean,f1_s0.6_mean,f1_s0.6_median,...,empty_rate_pct_s0.0,very_short_rate_pct_s0.0,substring_rate_pct_s0.25,empty_rate_pct_s0.25,very_short_rate_pct_s0.25,substring_rate_pct_s0.6,empty_rate_pct_s0.6,very_short_rate_pct_s0.6,pred_exact_match_rate_pct_s0.0_to_s0.25,pred_exact_match_rate_pct_s0.25_to_s0.6
0,0->0->0,2106,16.712977,15.384615,0.000000,15.538407,14.285714,0.000000,2.031343,0.000000,...,0.0,4.463438,11.253561,2.136752,19.040836,0.189934,0.000000,2.706553,4.938272,0.0
1,0->1->0,1965,20.635533,19.047619,0.000000,85.644285,100.000000,58.727735,2.567740,0.000000,...,0.0,1.323155,18.778626,0.000000,45.801527,0.152672,0.000000,1.679389,0.000000,0.0
2,1->1->0,1359,84.823568,100.000000,56.659308,87.796894,100.000000,61.295070,2.715922,0.000000,...,0.0,44.223694,18.469463,0.000000,46.504783,0.073584,0.073584,2.354673,17.586461,0.0
3,1->0->0,479,82.542025,100.000000,50.939457,15.759936,11.764706,0.000000,3.115182,0.000000,...,0.0,38.204593,13.778706,4.801670,29.645094,0.000000,0.000000,2.922756,0.000000,0.0
4,0->1->1,8,26.259926,34.814815,0.000000,77.391098,71.875000,25.000000,59.346591,57.272727,...,0.0,0.000000,25.000000,0.000000,25.000000,0.000000,0.000000,0.000000,0.000000,0.0
5,0->0->1,5,27.507792,24.000000,0.000000,33.117460,40.000000,0.000000,55.428571,57.142857,...,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,40.000000,0.0
6,1->1->1,5,78.095238,66.666667,40.000000,95.000000,100.000000,80.000000,53.409091,50.000000,...,0.0,40.000000,20.000000,0.000000,40.000000,0.000000,0.000000,0.000000,20.000000,0.0
7,1->0->1,1,66.666667,66.666667,0.000000,0.000000,0.000000,0.000000,50.000000,50.000000,...,0.0,100.000000,0.000000,0.000000,100.000000,0.000000,0.000000,0.000000,0.000000,0.0


In [30]:
def compact_transition_view(df_summary, transition_col_name):
    cols = [
        transition_col_name,
        "count",

        "f1_s0.0_mean", "f1_s0.25_mean", "f1_s0.6_mean",
        "exact_s0.0_mean", "exact_s0.25_mean", "exact_s0.6_mean",

        "pred_len_words_s0.0_mean", "pred_len_words_s0.25_mean", "pred_len_words_s0.6_mean",

        "gold_overlap_precision_s0.0_mean",
        "gold_overlap_precision_s0.25_mean",
        "gold_overlap_precision_s0.6_mean",

        "substring_rate_pct_s0.0",
        "substring_rate_pct_s0.25",
        "substring_rate_pct_s0.6",

        "pred_jaccard_s0.0_to_s0.25_mean",
        "pred_jaccard_s0.25_to_s0.6_mean",

        "pred_exact_match_rate_pct_s0.0_to_s0.25",
        "pred_exact_match_rate_pct_s0.25_to_s0.6",

        "empty_rate_pct_s0.0",
        "empty_rate_pct_s0.25",
        "empty_rate_pct_s0.6",
    ]

    cols = [c for c in cols if c in df_summary.columns]
    return df_summary[cols].copy()


print("=== Compact EXACT view ===")
summary_compact_exact = compact_transition_view(summary_exact_ext, "transition_exact")
display(summary_compact_exact)

print("=== Compact F1 view ===")
summary_compact_f1 = compact_transition_view(summary_f1_ext, "transition_f1")
display(summary_compact_f1)

=== Compact EXACT view ===


,transition_exact,count,f1_s0.0_mean,f1_s0.25_mean,f1_s0.6_mean,exact_s0.0_mean,exact_s0.25_mean,exact_s0.6_mean,pred_len_words_s0.0_mean,pred_len_words_s0.25_mean,...,substring_rate_pct_s0.0,substring_rate_pct_s0.25,substring_rate_pct_s0.6,pred_jaccard_s0.0_to_s0.25_mean,pred_jaccard_s0.25_to_s0.6_mean,pred_exact_match_rate_pct_s0.0_to_s0.25,pred_exact_match_rate_pct_s0.25_to_s0.6,empty_rate_pct_s0.0,empty_rate_pct_s0.25,empty_rate_pct_s0.6
0,0->0->0,3580,27.099896,33.379459,2.845462,0.0,0.0,0.0,20.933240,10.144972,...,20.670391,12.122905,0.139665,0.356357,0.128946,5.391061,0.0,0.0,1.592179,0.027933
1,0->1->0,1332,24.584342,100.000000,2.715796,0.0,100.0,0.0,20.526276,2.565315,...,13.813814,24.024024,0.150150,0.157737,0.061456,0.000000,0.0,0.0,0.000000,0.000000
2,1->1->0,661,100.000000,100.000000,1.683954,100.0,100.0,0.0,2.243570,2.205749,...,82.299546,19.667171,0.151286,0.959685,0.042216,23.146747,0.0,0.0,0.000000,0.000000
3,1->0->0,355,100.000000,29.129990,1.854647,100.0,0.0,0.0,2.515493,7.126761,...,78.873239,11.830986,0.000000,0.189547,0.123799,0.000000,0.0,0.0,3.098592,0.000000


=== Compact F1 view ===


,transition_f1,count,f1_s0.0_mean,f1_s0.25_mean,f1_s0.6_mean,exact_s0.0_mean,exact_s0.25_mean,exact_s0.6_mean,pred_len_words_s0.0_mean,pred_len_words_s0.25_mean,...,substring_rate_pct_s0.0,substring_rate_pct_s0.25,substring_rate_pct_s0.6,pred_jaccard_s0.0_to_s0.25_mean,pred_jaccard_s0.25_to_s0.6_mean,pred_exact_match_rate_pct_s0.0_to_s0.25,pred_exact_match_rate_pct_s0.25_to_s0.6,empty_rate_pct_s0.0,empty_rate_pct_s0.25,empty_rate_pct_s0.6
0,0->0->0,2106,16.712977,15.538407,2.031343,0.000000,0.000000,0.0,21.817664,11.003324,...,19.800570,11.253561,0.189934,0.345769,0.138585,4.938272,0.0,0.0,2.136752,0.000000
1,0->1->0,1965,20.635533,85.644285,2.567740,0.000000,58.727735,0.0,22.347583,4.304835,...,13.689567,18.778626,0.152672,0.177125,0.082799,0.000000,0.0,0.0,0.000000,0.000000
2,1->1->0,1359,84.823568,87.796894,2.715922,56.659308,61.295070,0.0,7.710081,5.640177,...,58.646063,18.469463,0.073584,0.759453,0.071001,17.586461,0.0,0.0,0.000000,0.073584
3,1->0->0,479,82.542025,15.759936,3.115182,50.939457,0.000000,0.0,8.392484,8.891441,...,54.697286,13.778706,0.000000,0.144190,0.120287,0.000000,0.0,0.0,4.801670,0.000000
4,0->1->1,8,26.259926,77.391098,59.346591,0.000000,25.000000,0.0,21.875000,9.000000,...,0.000000,25.000000,0.000000,0.299254,0.452056,0.000000,0.0,0.0,0.000000,0.000000
5,0->0->1,5,27.507792,33.117460,55.428571,0.000000,0.000000,0.0,18.000000,14.400000,...,20.000000,0.000000,0.000000,0.801374,0.308048,40.000000,0.0,0.0,0.000000,0.000000
6,1->1->1,5,78.095238,95.000000,53.409091,40.000000,80.000000,0.0,6.400000,4.400000,...,40.000000,20.000000,0.000000,0.673333,0.303333,20.000000,0.0,0.0,0.000000,0.000000
7,1->0->1,1,66.666667,0.000000,50.000000,0.000000,0.000000,0.0,2.000000,2.000000,...,0.000000,0.000000,0.000000,0.333333,0.000000,0.000000,0.0,0.0,0.000000,0.000000


In [31]:
interesting_groups = ["0->1->0", "1->1->0", "0->1->1", "1->1->1", "0->0->0"]

print("=== Interesting EXACT groups ===")
display(
    compact_transition_view(summary_exact_ext, "transition_exact")
    .query("transition_exact in @interesting_groups")
    .reset_index(drop=True)
)

print("=== Interesting F1 groups ===")
display(
    compact_transition_view(summary_f1_ext, "transition_f1")
    .query("transition_f1 in @interesting_groups")
    .reset_index(drop=True)
)

=== Interesting EXACT groups ===


,transition_exact,count,f1_s0.0_mean,f1_s0.25_mean,f1_s0.6_mean,exact_s0.0_mean,exact_s0.25_mean,exact_s0.6_mean,pred_len_words_s0.0_mean,pred_len_words_s0.25_mean,...,substring_rate_pct_s0.0,substring_rate_pct_s0.25,substring_rate_pct_s0.6,pred_jaccard_s0.0_to_s0.25_mean,pred_jaccard_s0.25_to_s0.6_mean,pred_exact_match_rate_pct_s0.0_to_s0.25,pred_exact_match_rate_pct_s0.25_to_s0.6,empty_rate_pct_s0.0,empty_rate_pct_s0.25,empty_rate_pct_s0.6
0,0->0->0,3580,27.099896,33.379459,2.845462,0.0,0.0,0.0,20.933240,10.144972,...,20.670391,12.122905,0.139665,0.356357,0.128946,5.391061,0.0,0.0,1.592179,0.027933
1,0->1->0,1332,24.584342,100.000000,2.715796,0.0,100.0,0.0,20.526276,2.565315,...,13.813814,24.024024,0.150150,0.157737,0.061456,0.000000,0.0,0.0,0.000000,0.000000
2,1->1->0,661,100.000000,100.000000,1.683954,100.0,100.0,0.0,2.243570,2.205749,...,82.299546,19.667171,0.151286,0.959685,0.042216,23.146747,0.0,0.0,0.000000,0.000000


=== Interesting F1 groups ===


,transition_f1,count,f1_s0.0_mean,f1_s0.25_mean,f1_s0.6_mean,exact_s0.0_mean,exact_s0.25_mean,exact_s0.6_mean,pred_len_words_s0.0_mean,pred_len_words_s0.25_mean,...,substring_rate_pct_s0.0,substring_rate_pct_s0.25,substring_rate_pct_s0.6,pred_jaccard_s0.0_to_s0.25_mean,pred_jaccard_s0.25_to_s0.6_mean,pred_exact_match_rate_pct_s0.0_to_s0.25,pred_exact_match_rate_pct_s0.25_to_s0.6,empty_rate_pct_s0.0,empty_rate_pct_s0.25,empty_rate_pct_s0.6
0,0->0->0,2106,16.712977,15.538407,2.031343,0.000000,0.000000,0.0,21.817664,11.003324,...,19.800570,11.253561,0.189934,0.345769,0.138585,4.938272,0.0,0.0,2.136752,0.000000
1,0->1->0,1965,20.635533,85.644285,2.567740,0.000000,58.727735,0.0,22.347583,4.304835,...,13.689567,18.778626,0.152672,0.177125,0.082799,0.000000,0.0,0.0,0.000000,0.000000
2,1->1->0,1359,84.823568,87.796894,2.715922,56.659308,61.295070,0.0,7.710081,5.640177,...,58.646063,18.469463,0.073584,0.759453,0.071001,17.586461,0.0,0.0,0.000000,0.073584
3,0->1->1,8,26.259926,77.391098,59.346591,0.000000,25.000000,0.0,21.875000,9.000000,...,0.000000,25.000000,0.000000,0.299254,0.452056,0.000000,0.0,0.0,0.000000,0.000000
4,1->1->1,5,78.095238,95.000000,53.409091,40.000000,80.000000,0.0,6.400000,4.400000,...,40.000000,20.000000,0.000000,0.673333,0.303333,20.000000,0.0,0.0,0.000000,0.000000


In [32]:
OUT_PATH_FULL = os.path.join(BASE_DIR, "transition_analysis_s0p00_s0p25_s0p60_enriched.csv")
OUT_PATH_EXACT_SUMMARY = os.path.join(BASE_DIR, "transition_summary_exact_enriched.csv")
OUT_PATH_F1_SUMMARY = os.path.join(BASE_DIR, "transition_summary_f1_enriched.csv")

df.to_csv(OUT_PATH_FULL, index=False)
summary_exact_ext.to_csv(OUT_PATH_EXACT_SUMMARY, index=False)
summary_f1_ext.to_csv(OUT_PATH_F1_SUMMARY, index=False)

print("Saved:", OUT_PATH_FULL)
print("Saved:", OUT_PATH_EXACT_SUMMARY)
print("Saved:", OUT_PATH_F1_SUMMARY)

Saved: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/analysis_Qwen2-1.5B-Instruct_pca_spike_first_token_hs/squad2_hasans_eval_all_layers_mlp_output/transition_analysis_s0p00_s0p25_s0p60_enriched.csv
Saved: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/analysis_Qwen2-1.5B-Instruct_pca_spike_first_token_hs/squad2_hasans_eval_all_layers_mlp_output/transition_summary_exact_enriched.csv
Saved: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/analysis_Qwen2-1.5B-Instruct_pca_spike_first_token_hs/squad2_hasans_eval_all_layers_mlp_output/transition_summary_f1_enriched.csv


In [33]:
OUT_PATH_EXACT_SUMMARY_COMPACT = os.path.join(BASE_DIR, "transition_summary_exact_enriched_compact.csv")
OUT_PATH_F1_SUMMARY_COMPACT = os.path.join(BASE_DIR, "transition_summary_f1_enriched_compact.csv")

summary_compact_exact.to_csv(OUT_PATH_EXACT_SUMMARY_COMPACT, index=False)
summary_compact_f1.to_csv(OUT_PATH_F1_SUMMARY_COMPACT, index=False)

print("Saved:", OUT_PATH_EXACT_SUMMARY_COMPACT)
print("Saved:", OUT_PATH_F1_SUMMARY_COMPACT)

Saved: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/analysis_Qwen2-1.5B-Instruct_pca_spike_first_token_hs/squad2_hasans_eval_all_layers_mlp_output/transition_summary_exact_enriched_compact.csv
Saved: /content/drive/MyDrive/TUM/Pratikum/SliceGPTExperiments/analysis_Qwen2-1.5B-Instruct_pca_spike_first_token_hs/squad2_hasans_eval_all_layers_mlp_output/transition_summary_f1_enriched_compact.csv


In [ ]:
######################
#let's go now deeper in the analysis looking into hs states